# 03 · Células de Perfil e Taxas de Desligamento

**Objetivo:** binning das dimensões, construção das células e cálculo das taxas por
motivo em cada nível de backoff, com suavização Empirical Bayes.

**Entradas:** `data/interim/rais/ano=*/**.parquet` (interim particionado).  
**Saídas:** `data/processed/rates/level_*.parquet` + `levels.json`;
`data/interim/val_<ano>.parquet` (agregado por ano p/ validação no nb06).

**Decisões centrais:**
- **Agregação incremental (map-reduce):** lê cada partição em lotes e SOMA as
  contagens por célula (`rates.Accumulator`). Nunca carrega tudo em memória —
  escala para cobertura nacional × múltiplos anos com RAM limitada.
- Exposição = nº de vínculos observados na célula no ano (denominador de risco).
- Taxa anual por motivo = desligamentos do motivo / exposição.
- Backoff hierárquico (`src/cells.py`) do mais geral ao mais específico.
- Shrinkage Beta-Binomial aninhado resolve células pouco populosas.

**Limitações:** viés de composição; taxa é risco *médio* da célula, não individual.

In [ ]:
# Preâmbulo: torna o pacote src importável a partir do notebook
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from src.config import load_config, anos_validos
cfg = load_config()
print('Raiz:', cfg['root'], '| modo sintético:', cfg['synthetic_mode'])

In [ ]:
import pyarrow.parquet as pq
from src import binning, cells, rates
interim = cfg['abs']['interim']
motivos = cfg['motivos']
anos_ok = set(anos_validos(cfg))   # respeita exclusão de anos atípicos

# Nível usado na validação temporal (populoso e estável)
VAL_LVL = 'cbo2_cnae2_tempo_uf'
val_cols = [l['cols'] for l in cells.BACKOFF_LEVELS if l['name']==VAL_LVL][0]

niveis = cells.active_levels(cfg)           # exclui níveis ultra-granulares (config)
print('níveis materializados:', [l['name'] for l in niveis])
acc = rates.Accumulator(motivos, levels=niveis)   # acumulador global (todos os anos)
val_by_year = {}                            # acumulador do nível de validação por ano
files = sorted((interim / 'rais').rglob('*.parquet'))
print('partições a processar:', len(files))

In [ ]:
# Passe incremental: lote a lote, sem materializar os microdados inteiros
for f in files:
    ano = int(f.parent.name.split('=')[1])
    if ano not in anos_ok: continue
    pf = pq.ParquetFile(f)
    for batch in pf.iter_batches(batch_size=3_000_000):
        df = batch.to_pandas()
        df = cells.add_cell_keys(binning.add_bins(df, cfg))
        acc.add(df)                                       # soma em todos os níveis
        v = rates.count_single_level(df, val_cols, motivos)
        val_by_year[ano] = rates._sum_two_tables(val_by_year.get(ano), v, val_cols, motivos)
    print(f'  ok {f.parent.name}/{f.name}', flush=True)
tables = acc.tables()
for name, tab in tables.items():
    print(f'{name:32s} células={len(tab):>9,}  exposição_total={int(tab["n"].sum()):,}')

In [ ]:
# Persistir tabelas de taxa (scoring) + agregados por ano (validação)
rates.save_level_tables(tables, motivos, cfg['abs']['rates'])
for ano, v in val_by_year.items():
    v.to_parquet(interim / f'val_{ano}.parquet', index=False)
print('Tabelas salvas em', cfg['abs']['rates'], '| anos validação:', sorted(val_by_year))

# Espiar o nível mais específico com a taxa involuntária bruta (k/n)
comp = tables['completo'].copy()
comp['taxa_sjc_bruta'] = comp['k_involuntario_sjc'] / comp['n']
display(comp.sort_values('n', ascending=False).head(8))

> **Nota metodológica:** a taxa *bruta* (k/n) acima é apenas diagnóstica.
> A taxa usada no score é a versão **suavizada** (Empirical Bayes aninhado),
> calculada sob demanda em `src/rates.eb_annual_hazard` durante o scoring —
> assim cada consulta recua na hierarquia exatamente até onde há suporte estatístico.